In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Load the two pickle files
df1 = pd.read_pickle("eggs_y_norm.pkl")
df2 = pd.read_pickle("../tests/data/eggs_y_norm.pkl")  # Change to your second file

print(f"DataFrame 1 loaded: {df1.shape[0]} rows, {df1.shape[1]} columns")
print(f"DataFrame 2 loaded: {df2.shape[0]} rows, {df2.shape[1]} columns")

In [3]:
# Compare shapes
print("=== Shape Comparison ===")
print(f"DF1 shape: {df1.shape}")
print(f"DF2 shape: {df2.shape}")
print(f"Difference: {df1.shape[0] - df2.shape[0]} rows, {df1.shape[1] - df2.shape[1]} columns")

In [4]:
# Compare columns
print("=== Column Comparison ===")
cols1 = set(df1.columns)
cols2 = set(df2.columns)

print(f"Columns in DF1 only: {cols1 - cols2}")
print(f"Columns in DF2 only: {cols2 - cols1}")
print(f"Common columns: {len(cols1 & cols2)}")

In [5]:
# Compare data types for common columns
print("=== Data Type Comparison ===")
common_cols = list(cols1 & cols2)
dtype_diff = []

for col in common_cols:
    if df1[col].dtype != df2[col].dtype:
        dtype_diff.append((col, df1[col].dtype, df2[col].dtype))

if dtype_diff:
    print("Columns with different data types:")
    for col, dt1, dt2 in dtype_diff:
        print(f"  {col}: {dt1} vs {dt2}")
else:
    print("All common columns have the same data types")

In [6]:
# Compare basic statistics for numeric columns
print("=== Numeric Column Statistics Comparison ===")
numeric_cols = [col for col in common_cols if pd.api.types.is_numeric_dtype(df1[col])]

comparison_stats = pd.DataFrame({
    'DF1_mean': df1[numeric_cols].mean(),
    'DF2_mean': df2[numeric_cols].mean(),
    'DF1_std': df1[numeric_cols].std(),
    'DF2_std': df2[numeric_cols].std(),
    'DF1_min': df1[numeric_cols].min(),
    'DF2_min': df2[numeric_cols].min(),
    'DF1_max': df1[numeric_cols].max(),
    'DF2_max': df2[numeric_cols].max()
})

comparison_stats['mean_diff'] = comparison_stats['DF1_mean'] - comparison_stats['DF2_mean']
print(comparison_stats)

In [7]:
# Inspect differences in prev2_rates
print("=== Differences in prev2_rates ===")
diff_mask_prev2 = df1['prev2_rates'] != df2['prev2_rates']
print(f"Number of differing rows: {diff_mask_prev2.sum()}")
print("\nSample differences:")
print(df1[diff_mask_prev2][['prev2_rates']].head(10).rename(columns={'prev2_rates': 'DF1_prev2_rates'})
      .join(df2[diff_mask_prev2][['prev2_rates']].head(10).rename(columns={'prev2_rates': 'DF2_prev2_rates'})))

# Show the absolute and relative differences
print("\n=== Difference statistics ===")
abs_diff = (df1[diff_mask_prev2]['prev2_rates'] - df2[diff_mask_prev2]['prev2_rates']).abs()
print(f"Mean absolute difference: {abs_diff.mean():.6f}")
print(f"Max absolute difference: {abs_diff.max():.6f}")
print(f"Min absolute difference: {abs_diff.min():.6f}")

In [8]:
# Check NaN values in prev2_rates
print("=== NaN Analysis for prev2_rates ===")
nan_in_df1_prev2 = df1['prev2_rates'].isna().sum()
nan_in_df2_prev2 = df2['prev2_rates'].isna().sum()
print(f"NaN count in DF1: {nan_in_df1_prev2}")
print(f"NaN count in DF2: {nan_in_df2_prev2}")

# Check where NaNs differ between the two dataframes
nan_mask_df1_prev2 = df1['prev2_rates'].isna()
nan_mask_df2_prev2 = df2['prev2_rates'].isna()
nan_only_in_df1_prev2 = nan_mask_df1_prev2 & ~nan_mask_df2_prev2
nan_only_in_df2_prev2 = ~nan_mask_df1_prev2 & nan_mask_df2_prev2
both_nan_prev2 = nan_mask_df1_prev2 & nan_mask_df2_prev2

print(f"\nNaN only in DF1: {nan_only_in_df1_prev2.sum()}")
print(f"NaN only in DF2: {nan_only_in_df2_prev2.sum()}")
print(f"NaN in both: {both_nan_prev2.sum()}")

# Check actual numeric differences (excluding NaN)
print("\n=== Numeric Differences (excluding NaN) ===")
numeric_diff_mask_prev2 = ~nan_mask_df1_prev2 & ~nan_mask_df2_prev2 & (df1['prev2_rates'] != df2['prev2_rates'])
print(f"Rows with numeric differences: {numeric_diff_mask_prev2.sum()}")

if numeric_diff_mask_prev2.sum() > 0:
    abs_diff_numeric_prev2 = (df1[numeric_diff_mask_prev2]['prev2_rates'] - df2[numeric_diff_mask_prev2]['prev2_rates']).abs()
    print(f"Mean absolute difference: {abs_diff_numeric_prev2.mean():.6f}")
    print(f"Max absolute difference: {abs_diff_numeric_prev2.max():.6f}")
    print(f"Min absolute difference: {abs_diff_numeric_prev2.min():.6f}")
    print("\nSample numeric differences:")
    print(df1[numeric_diff_mask_prev2][['prev2_rates']].head(10).rename(columns={'prev2_rates': 'DF1_prev2_rates'})
          .join(df2[numeric_diff_mask_prev2][['prev2_rates']].head(10).rename(columns={'prev2_rates': 'DF2_prev2_rates'})))

In [ ]:
# Check for missing values comparison
print("=== Missing Values Comparison ===")
missing_comparison = pd.DataFrame({
    'DF1_missing': df1[common_cols].isnull().sum(),
    'DF2_missing': df2[common_cols].isnull().sum(),
    'DF1_missing_pct': (df1[common_cols].isnull().sum() / len(df1) * 100).round(2),
    'DF2_missing_pct': (df2[common_cols].isnull().sum() / len(df2) * 100).round(2)
})

print(missing_comparison[missing_comparison['DF1_missing'] + missing_comparison['DF2_missing'] > 0])

In [11]:
# If dataframes have same index/key, compare values directly
# Assuming 'id_trap' and 'end_date' can be used as keys
if 'id_trap' in common_cols and 'end_date' in common_cols:
    print("=== Direct Value Comparison (on common keys) ===")
    df1_indexed = df1.set_index(['id_trap', 'end_date'])
    df2_indexed = df2.set_index(['id_trap', 'end_date'])
    
    common_index = df1_indexed.index.intersection(df2_indexed.index)
    print(f"Common rows: {len(common_index)}")
    
    if len(common_index) > 0:
        # Compare a specific column for common rows
        if 'weeklyRates' in common_cols:
            diff = df1_indexed.loc[common_index, 'weeklyRates'] - df2_indexed.loc[common_index, 'weeklyRates']
            print(f"\nweeklyRates differences (for common rows):")
            print(f"  Non-zero differences: {(diff != 0).sum()}")
            print(f"  Max absolute difference: {abs(diff).max()}")

In [12]:
# Visualize distributions of a key column
if 'weeklyRates' in common_cols:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(df1['weeklyRates'].dropna(), bins=50, alpha=0.7, label='DF1')
    axes[0].hist(df2['weeklyRates'].dropna(), bins=50, alpha=0.7, label='DF2')
    axes[0].set_xlabel('weeklyRates')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution Comparison')
    axes[0].legend()
    
    axes[1].boxplot([df1['weeklyRates'].dropna(), df2['weeklyRates'].dropna()], labels=['DF1', 'DF2'])
    axes[1].set_ylabel('weeklyRates')
    axes[1].set_title('Box Plot Comparison')
    
    plt.tight_layout()
    plt.show()

In [13]:
# Summary report
print("=== SUMMARY ===")
print(f"Total rows: DF1={df1.shape[0]}, DF2={df2.shape[0]}")
print(f"Total columns: DF1={df1.shape[1]}, DF2={df2.shape[1]}")
print(f"Common columns: {len(common_cols)}")
print(f"Memory usage: DF1={df1.memory_usage(deep=True).sum() / 1024**2:.2f} MB, DF2={df2.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
list(df1.columns)

In [ ]:
list(df2.columns)

In [ ]:
#check if the columns "prev1_weeklyRates" of df1 is equal to "prev1_rates" of df2
df1['prev_weeklyRates'].equals(df2['prev1_rates'])
# check how many values are different
(df1['prev_weeklyRates'] != df2['prev1_rates']).sum()
# show the different values
df1[df1['prev_weeklyRates'] != df2['prev1_rates']][['prev_weeklyRates']].join(df2[df1['prev_weeklyRates'] != df2['prev1_rates']][['prev1_rates']])

In [ ]:
df2['prev1_rates']